Import & Path

In [16]:
import sys
print(sys.executable)

/opt/miniconda3/envs/ee519_arm/bin/python


In [33]:
import sys
!{sys.executable} -m pip install -q ipynbname
!{sys.executable} -m pip install -q -U openai-whisper
!{sys.executable} -m pip install -q sounddevice soundfile

In [19]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import Audio, display
import os
from pathlib import Path
import ipynbname

In [20]:
nb_path = Path(ipynbname.path()).resolve()   # .../SwitchNet/notebooks/demo.ipynb
ROOT = nb_path.parent.parent                 # .../SwitchNet
os.chdir(ROOT)

print("Notebook:", nb_path)
print("ROOT:", ROOT)
print("CWD:", Path.cwd())

Notebook: /Users/william/Document/EE/EE519/Team_Proj/SwitchNet/notebooks/demo.ipynb
ROOT: /Users/william/Document/EE/EE519/Team_Proj/SwitchNet
CWD: /Users/william/Document/EE/EE519/Team_Proj/SwitchNet


In [21]:
DATA_DIR  = ROOT / "data" / "LibriSpeech" / "test-clean" / "61" / "70968"
TRANS_PATH = DATA_DIR / "61-70968.trans.txt"
EVAL_DIR  = ROOT / "eval_outputs"

print("DATA_DIR exists?:", DATA_DIR.exists())
print("TRANS exists?:", TRANS_PATH.exists())
print("EVAL_DIR exists?:", EVAL_DIR.exists())

DATA_DIR exists?: True
TRANS exists?: True
EVAL_DIR exists?: True


List a few utterances

In [23]:
flacs = sorted(DATA_DIR.glob("*.flac"))
print("Num flac:", len(flacs))
flacs[:10]

Num flac: 63


[PosixPath('/Users/william/Document/EE/EE519/Team_Proj/SwitchNet/data/LibriSpeech/test-clean/61/70968/61-70968-0000.flac'),
 PosixPath('/Users/william/Document/EE/EE519/Team_Proj/SwitchNet/data/LibriSpeech/test-clean/61/70968/61-70968-0001.flac'),
 PosixPath('/Users/william/Document/EE/EE519/Team_Proj/SwitchNet/data/LibriSpeech/test-clean/61/70968/61-70968-0002.flac'),
 PosixPath('/Users/william/Document/EE/EE519/Team_Proj/SwitchNet/data/LibriSpeech/test-clean/61/70968/61-70968-0003.flac'),
 PosixPath('/Users/william/Document/EE/EE519/Team_Proj/SwitchNet/data/LibriSpeech/test-clean/61/70968/61-70968-0004.flac'),
 PosixPath('/Users/william/Document/EE/EE519/Team_Proj/SwitchNet/data/LibriSpeech/test-clean/61/70968/61-70968-0005.flac'),
 PosixPath('/Users/william/Document/EE/EE519/Team_Proj/SwitchNet/data/LibriSpeech/test-clean/61/70968/61-70968-0006.flac'),
 PosixPath('/Users/william/Document/EE/EE519/Team_Proj/SwitchNet/data/LibriSpeech/test-clean/61/70968/61-70968-0007.flac'),
 PosixPa

Load transcripts

In [24]:
ref = {}
with open(TRANS_PATH, "r", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split()
        if not parts: 
            continue
        utt_id = parts[0]
        text = " ".join(parts[1:])
        ref[utt_id] = text

# check
some_id = flacs[0].stem
print(some_id)
print(ref.get(some_id, "NOT FOUND"))

61-70968-0000
HE BEGAN A CONFUSED COMPLAINT AGAINST THE WIZARD WHO HAD VANISHED BEHIND THE CURTAIN ON THE LEFT


Pick 3 examples and play audio

In [25]:
example_ids = [flacs[0].stem, flacs[1].stem, flacs[2].stem]

for uid in example_ids:
    audio_path = DATA_DIR / f"{uid}.flac"
    print("\n=== ", uid, " ===")
    print("REF:", ref.get(uid, ""))
    display(Audio(filename=str(audio_path)))


===  61-70968-0000  ===
REF: HE BEGAN A CONFUSED COMPLAINT AGAINST THE WIZARD WHO HAD VANISHED BEHIND THE CURTAIN ON THE LEFT



===  61-70968-0001  ===
REF: GIVE NOT SO EARNEST A MIND TO THESE MUMMERIES CHILD



===  61-70968-0002  ===
REF: A GOLDEN FORTUNE AND A HAPPY LIFE


Run Whisper on one file (English)

In [28]:
import whisper

model = whisper.load_model("large-v3") # choose model

100%|█████████████████████████████████████| 2.88G/2.88G [01:56<00:00, 26.5MiB/s]


In [29]:
uid = example_ids[0]
audio_path = DATA_DIR / f"{uid}.flac"

hyp = model.transcribe(
    str(audio_path),
    language="en",
    task="transcribe",
    fp16=False
)

print("HYP:", hyp["text"])

HYP:  He began a confused complaint against the wizard, who had vanished behind the curtain on the left.


Show segments table

In [30]:
seg_df = pd.DataFrame([{
    "start": s["start"],
    "end": s["end"],
    "text": s["text"].strip()
} for s in hyp["segments"]])

seg_df

,start,end,text
0,0.0,4.0,He began a confused complaint against the wiza...
1,4.0,4.9,on the left.


Compute WER (simple)

In [31]:
def wer(ref_text, hyp_text):
    r = ref_text.split()
    h = hyp_text.split()
    # classic DP edit distance
    dp = [[0]*(len(h)+1) for _ in range(len(r)+1)]
    for i in range(len(r)+1):
        dp[i][0] = i
    for j in range(len(h)+1):
        dp[0][j] = j
    for i in range(1, len(r)+1):
        for j in range(1, len(h)+1):
            cost = 0 if r[i-1] == h[j-1] else 1
            dp[i][j] = min(
                dp[i-1][j] + 1,      # deletion
                dp[i][j-1] + 1,      # insertion
                dp[i-1][j-1] + cost  # substitution
            )
    return dp[-1][-1] / max(1, len(r))

uid = example_ids[0]
ref_text = ref[uid]
hyp_text = hyp["text"].strip()

print("WER:", wer(ref_text, hyp_text))
print("REF:", ref_text)
print("HYP:", hyp_text)

WER: 1.0
REF: HE BEGAN A CONFUSED COMPLAINT AGAINST THE WIZARD WHO HAD VANISHED BEHIND THE CURTAIN ON THE LEFT
HYP: He began a confused complaint against the wizard, who had vanished behind the curtain on the left.


(Optional)Show the result

In [32]:
csv_path = EVAL_DIR / "largev3_61-70968.csv"
df = pd.read_csv(csv_path)
df.head()

,utt_id,wer,audio,ref,hyp
0,61-70968-0000,0.0,data/LibriSpeech/test-clean/61/70968/61-70968-...,HE BEGAN A CONFUSED COMPLAINT AGAINST THE WIZA...,he began a confused complaint against the wiza...
1,61-70968-0001,0.0,data/LibriSpeech/test-clean/61/70968/61-70968-...,GIVE NOT SO EARNEST A MIND TO THESE MUMMERIES ...,"Give not so earnest a mind to these mummeries,..."
2,61-70968-0002,0.0,data/LibriSpeech/test-clean/61/70968/61-70968-...,A GOLDEN FORTUNE AND A HAPPY LIFE,A golden fortune and a happy life.
3,61-70968-0003,0.0,data/LibriSpeech/test-clean/61/70968/61-70968-...,HE WAS LIKE UNTO MY FATHER IN A WAY AND YET WA...,he was like unto my father in a way and yet wa...
4,61-70968-0004,0.0,data/LibriSpeech/test-clean/61/70968/61-70968-...,ALSO THERE WAS A STRIPLING PAGE WHO TURNED INT...,also there was a stripling page who turned int...


B. Record your own

In [34]:
import sounddevice as sd
import soundfile as sf
import numpy as np
from pathlib import Path
from IPython.display import Audio, display

def record_audio(duration_s=6.0, fs=16000, channels=1):
    """Record audio from default microphone. Returns mono float32 signal."""
    print(f"Recording {duration_s:.1f}s... speak now!")
    x = sd.rec(int(duration_s * fs), samplerate=fs, channels=channels, dtype="float32")
    sd.wait()
    x = x.squeeze()
    print("Done.")
    return x, fs

# record
x, fs = record_audio(duration_s=6, fs=16000)

# playback
display(Audio(x, rate=fs))

# save (optional but recommended)
out_wav = Path("outputs/live_demo.wav")
out_wav.parent.mkdir(parents=True, exist_ok=True)
sf.write(out_wav, x, fs)
print("Saved:", out_wav)

Recording 6.0s... speak now!
Done.


/opt/miniconda3/envs/ee519_arm/lib/python3.11/site-packages/IPython/lib/display.py:188: RuntimeWarning: invalid value encountered in divide
  scaled = data / normalization_factor * 32767
/opt/miniconda3/envs/ee519_arm/lib/python3.11/site-packages/IPython/lib/display.py:189: RuntimeWarning: invalid value encountered in cast
  return scaled.astype("<h").tobytes(), nchan


Saved: outputs/live_demo.wav


Use Whisper

In [35]:
import whisper
model = whisper.load_model("small")  # samller model for faster

res = model.transcribe(str(out_wav), language="en", task="transcribe", fp16=False)
hyp_text = res["text"].strip()
print("HYP:", hyp_text)

100%|███████████████████████████████████████| 461M/461M [00:22<00:00, 21.2MiB/s]


HYP: you


Calculate the WER

In [ ]:
ref_text = "THIS IS A FIXED SENTENCE".strip() #change this

def simple_norm(s: str) -> str:
    return " ".join(s.lower().split())

def wer(ref, hyp):
    r = simple_norm(ref).split()
    h = simple_norm(hyp).split()
    dp = [[0]*(len(h)+1) for _ in range(len(r)+1)]
    for i in range(len(r)+1): dp[i][0] = i
    for j in range(len(h)+1): dp[0][j] = j
    for i in range(1, len(r)+1):
        for j in range(1, len(h)+1):
            cost = 0 if r[i-1] == h[j-1] else 1
            dp[i][j] = min(dp[i-1][j]+1, dp[i][j-1]+1, dp[i-1][j-1]+cost)
    return dp[-1][-1] / max(1, len(r))

print("REF:", ref_text)
print("WER:", wer(ref_text, hyp_text))

REF: THIS IS A FIXED SENTENCE I READ FOR THE DEMO
WER: 1.0
